In [2]:
import numpy as np
import arviz
import x3cflux
import hopsy
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
data = x3cflux.FluxMLParser().parse("simple_model.fml")
print([config.name for config in data.configurations])


['freeflux']


In [4]:
config = data.configurations[0]
simulator = x3cflux.create_simulator_from_data(data.network_data, config, sim_method="auto")
print(simulator.parameter_space.free_parameter_names)


['R2.n', 'E_out.n', 'F_out.n']


In [ ]:
# get_parameters() reads <fluxvalue> entries from the FML <simulation> block.
# The drain reactions E_out / F_out have no entry there, so x3cflux sets those
# free parameters to 0 and warns. A zero F_out -> metabolite F carries no flux
# -> EMU solver is singular (loss = NaN).  Use the grid search below instead.
params = x3cflux.get_parameters(
    simulator.parameter_space, simulator.configurations[0].parameter_entries
)
print("Initial params from FML (may contain zeros):", params)

In [ ]:
# Inspect the flux polytope:  A*x <= b  (columns = free-parameter order)
ineq_sys = simulator.parameter_space.inequality_system
A = np.array(ineq_sys.matrix)
b = np.array(ineq_sys.bound)
print("A (inequality matrix):\n", A)
print("b (rhs bounds):", b)
# NOTE: run_uniform_sampling() uses hopsy/PolyRound internally.
# It fails here because the E_out upper bound (~300) creates a scale mismatch
# that breaks the Maximum Volume Ellipsoid Cholesky factorisation -> NaN/Inf.
# We replace it with a feasibility-checking grid search (see cell below).

In [ ]:
def compare_meas(simulator, ps):
    """Bar chart: real vs simulated MDV at parameter vector ps."""
    names     = simulator.measurement_names[0]
    real_meas = simulator.measurement_data[0]
    real_sd   = simulator.measurement_standard_deviations[0]
    sim_meas  = simulator.compute_measurements(ps)[0]
    width = 0.35
    for m in range(len(names)):
        n = len(real_meas[m])
        x = np.arange(n)
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(x - width/2, real_meas[m], width, label="Real",
               yerr=real_sd[m], capsize=5, color="tab:blue")
        ax.bar(x + width/2, sim_meas[m],  width, label="Sim", color="tab:orange")
        ax.set_xlabel("Mass shift")
        ax.set_ylabel("Fractional labeling enrichment")
        ax.set_title(names[m])
        ax.set_xticks(x)
        ax.set_xticklabels([f"M+{i}" for i in range(n)])
        ax.set_ylim([0, 1])
        ax.legend()
        plt.show()

In [ ]:
def compare_meas(simulator, ps):
    names = simulator.measurement_names[0]
    real_meas = simulator.measurement_data[0]
    real_meas_stddev = simulator.measurement_standard_deviations[0]
    sim_meas = simulator.compute_measurements(ps)[0]
    width = 0.35      # width of the bars

    for m in range(len(names)):
        n = len(real_meas[m])
        x = np.arange(n)  # the label locations

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(x - width/2, real_meas[m], width, label='Real', yerr=real_meas_stddev[m], capsize=5, color='tab:blue')
        ax.bar(x + width/2, sim_meas[m], width, label='Sim', color='tab:orange')

        ax.set_xlabel('Mass shift')
        ax.set_ylabel('Fractonal labeling enrichment')
        ax.set_title(f'{names[m]}')
        ax.set_xticks(x)
        plt.ylim([0, 1])
        ax.set_xticklabels([f"M+{i}" for i in range(n)])
        ax.legend()
        plt.show()

In [ ]:
# Use a manually-chosen feasible starting point (Ax<=b, finite loss):
# R2=20, E_out=60, F_out=10 -- all constraints satisfied, F carries non-zero flux.
p0 = np.array([20.0, 60.0, 10.0])
print(f"Residual at p0={p0}: {simulator.compute_loss(p0):.4g}")
compare_meas(simulator, p0)

In [ ]:
# run_uniform_sampling crashes (see note above).
# Build starting points via a stoichiometry-aware grid search instead.
def feasible_starting_points(sim, n_grid=5):
    """Return (n_free_params, n_starts) array of feasible, finite-loss points."""
    ps = sim.parameter_space
    A  = np.array(ps.inequality_system.matrix)
    b  = np.array(ps.inequality_system.bound)
    n  = ps.num_free_parameters
    upper = np.full(n, 100.0)
    for i in range(n):
        col = A[:, i]
        pos = col > 0
        if pos.any():
            upper[i] = min(upper[i], np.min(b[pos] / col[pos]))
    grids = [np.linspace(1.0, float(u) * 0.95, n_grid) for u in upper]
    candidates = []
    for vals in np.array(np.meshgrid(*grids, indexing="ij")).reshape(n, -1).T:
        if np.all(A @ vals <= b) and np.isfinite(sim.compute_loss(vals)):
            candidates.append(vals)
    if not candidates:
        raise RuntimeError("No feasible starting points found.")
    return np.array(candidates).T   # shape: (n_free_params, n_starts)

samples = feasible_starting_points(simulator, n_grid=5)
print(f"Feasible starting points: {samples.shape[1]}  (shape {samples.shape})")

In [ ]:
# run_multi_optimization expects starting_points of shape (n_free_params, n_starts)
optima, obj_val = x3cflux.run_multi_optimization(
    simulator,
    starting_points=samples,   # (n_free_params, n_starts) <- correct orientation
    max_iter=1000,
    tol=1e-8,
    num_procs=1,               # keep 1 -- parallel fork breaks in notebooks
)
# optima shape is also (n_free_params, n_starts)
sorted_idx = np.argsort(obj_val)
print("Top 5 optima:")
for i in range(min(5, len(obj_val))):
    opt = optima[:, sorted_idx[i]]
    print(f"  {dict(zip(simulator.parameter_space.free_parameter_names, opt))}  loss={obj_val[sorted_idx[i]]:.4f}")

In [ ]:
best_idx = np.argmin(obj_val)
optimum  = optima[:, best_idx]
print(f"Best optimum: {dict(zip(simulator.parameter_space.free_parameter_names, optimum))}")
print(f"Residual    : {obj_val[best_idx]:.4f}")
compare_meas(simulator, optimum)

In [ ]:
free_names = simulator.parameter_space.free_parameter_names  # ['R2.n', 'E_out.n', 'F_out.n']
alpha = 0.95
pl_intervals = x3cflux.run_profile_likelihood_cis(
    simulator, optimum, alpha=alpha, names=free_names, num_procs=1
)
print(f"{int(100*alpha)}% profile-likelihood confidence intervals:")
for name, (lo, hi) in zip(free_names, pl_intervals):
    opt = optimum[free_names.index(name)]
    print(f"  {name:15s}: [{lo:.3f}, {hi:.3f}]  (optimum = {opt:.3f})")

## Confidence intervals for all fluxes

x3cflux chose `[R2, E_out, F_out]` as free parameters, but the true DOF of this
network is **2** — the polytope rows 6–9 encode the stoichiometric constraint
`F_out = 100 - E_out/3`, so the three "free" params live on a 2-D manifold.

A cleaner 2-D parameterisation is **[R3, R5]** (the two biologically meaningful
internal fluxes).  All other fluxes follow from steady-state mass balance:

| Flux | Expression |
|------|-----------|
| R1   | 100 (measured) |
| R2   | 100 + R3 − 2·R5 |
| R4   | R5  (C balance) |
| R6   | 100 − R5 |
| E_out | 3·R5 |
| F_out | 100 − R5 |

The strategy:
1. Reparameterise from `[R2, E_out, F_out]` (x3cflux) to `[R3, R5]` via the
   change-of-basis Jacobian **M** (2×3).
2. Transform the 3×3 covariance → 2×2 covariance in `[R3, R5]` space.
3. Propagate to all 8 fluxes using the 8×2 Jacobian **J**.

In [ ]:
import scipy.stats

# ── Analytical reparameterisation ────────────────────────────────────────────
# From mass balance with R1=100:
#   R5 = E_out / 3
#   R3 = R2 + E_out/3 - F_out   (D balance: R2 + R5 = R3 + R6 = R3 + F_out)

def xcflux_to_r3r5(v):
    """[R2, E_out, F_out] -> [R3, R5]."""
    R2, E_out, F_out = v
    R5 = E_out / 3.0
    R3 = R2 + R5 - F_out        # from D balance
    return np.array([R3, R5])

def r3r5_to_xcflux(R3, R5):
    """[R3, R5] -> x3cflux free params [R2, E_out, F_out]."""
    return np.array([100 + R3 - 2*R5, 3*R5, 100 - R5])

def all_net_fluxes(R3, R5):
    """All 8 net fluxes from (R3, R5). Order matches parameter_names .n entries."""
    return np.array([
        100.0,          # R1
        100+R3-2*R5,    # R2
        R3,             # R3
        R5,             # R4
        R5,             # R5
        100-R5,         # R6
        3*R5,           # E_out
        100-R5,         # F_out
    ])

# ── Verify reparameterisation matches the x3cflux optimum ────────────────────
R3_opt, R5_opt = xcflux_to_r3r5(optimum)
print(f"Optimum in [R3, R5] space:  R3={R3_opt:.4f}  R5={R5_opt:.4f}")
print(f"Round-trip check (should match optimum):")
print(f"  x3cflux -> [R3,R5] -> x3cflux : {r3r5_to_xcflux(R3_opt, R5_opt)}")
print(f"  original optimum               : {optimum}")

In [ ]:
# ── Compute covariance in x3cflux free-param space first ─────────────────────
cov_raw, non_ident = x3cflux.compute_free_parameter_covariance(simulator, optimum)
cov = np.array(cov_raw)
print("Non-identifiable params:", non_ident or "none")
print("3×3 covariance (R2, E_out, F_out):")
print(cov)
print()

# ── Change-of-basis Jacobian  M : d[R3, R5] / d[R2, E_out, F_out]  (2×3) ───
# R3 = R2 + E_out/3 - F_out  =>  dR3/dR2=1, dR3/dE_out=1/3, dR3/dF_out=-1
# R5 = E_out / 3              =>  dR5/dR2=0, dR5/dE_out=1/3, dR5/dF_out=0
M = np.array([
    [1, 1/3, -1],   # dR3
    [0, 1/3,  0],   # dR5
])  # shape (2, 3)

# ── Covariance in [R3, R5] space ─────────────────────────────────────────────
cov_free_arr = np.array(cov)       # 3×3  (from compute_free_parameter_covariance)
cov_r3r5     = M @ cov_free_arr @ M.T   # 2×2
std_r3r5     = np.sqrt(np.diag(cov_r3r5))

print("Covariance in [R3, R5] space:")
print(cov_r3r5)
print()
print(f"Linearised std: R3 ± {std_r3r5[0]:.3f}  |  R5 ± {std_r3r5[1]:.3f}")

# ── All-flux Jacobian  J : d(all_net_fluxes) / d[R3, R5]  (8×2) ─────────────
# Columns [dR3, dR5]; rows match net-flux names R1..F_out
J = np.array([
    [0,  0],   # R1    (fixed)
    [1, -2],   # R2  = 100 + R3 - 2*R5
    [1,  0],   # R3
    [0,  1],   # R4  = R5
    [0,  1],   # R5
    [0, -1],   # R6  = 100 - R5
    [0,  3],   # E_out = 3*R5
    [0, -1],   # F_out = 100 - R5
])  # shape (8, 2)

cov_all_net = J @ cov_r3r5 @ J.T   # 8×8
std_all_net = np.sqrt(np.diag(cov_all_net))

net_names = [n for n in simulator.parameter_space.parameter_names if n.endswith(".n")]
net_opt   = all_net_fluxes(R3_opt, R5_opt)
print()
print("All-flux linearised SDs:")
for name, val, sd in zip(net_names, net_opt, std_all_net):
    print(f"  {name:12s}: {val:8.3f}  ±  {sd:.3f}")

In [ ]:
# ── Profile-likelihood CIs for R3 and R5 ─────────────────────────────────────
# Fix one parameter, minimise loss over the other, find the chi-squared boundary.
from scipy.optimize import minimize_scalar, brentq

loss_opt_val = simulator.compute_loss(optimum)
alpha  = 0.95
thresh = scipy.stats.chi2.ppf(alpha, df=1) / 2   # one-parameter PL threshold

def profile_loss(fixed_val, fixed_is_R3, bounds_other=(0, 100)):
    """Profile loss: fix one of R3/R5, optimise over the other."""
    def obj(other):
        R3, R5 = (fixed_val, other) if fixed_is_R3 else (other, fixed_val)
        free = r3r5_to_xcflux(R3, R5)
        ps_  = simulator.parameter_space
        A_, b_ = np.array(ps_.inequality_system.matrix), np.array(ps_.inequality_system.bound)
        if not np.all(A_ @ free <= b_ + 1e-6):
            return 1e9
        val = simulator.compute_loss(free)
        return val if np.isfinite(val) else 1e9
    res = minimize_scalar(obj, bounds=bounds_other, method="bounded")
    return res.fun - loss_opt_val

pl_cis = {}
for name, fixed_val_opt, fixed_is_R3, bounds_scan, bounds_other in [
    ("R3", R3_opt, True,  (0, 100), (0, 100)),
    ("R5", R5_opt, False, (0, 100), (0, 100)),
]:
    # Lower bound: scan below optimum
    try:
        lo = brentq(lambda v: profile_loss(v, fixed_is_R3, bounds_other) - thresh,
                    0.01, fixed_val_opt - 0.01)
    except (ValueError, RuntimeError):
        lo = 0.0

    # Upper bound: scan above optimum
    try:
        hi = brentq(lambda v: profile_loss(v, fixed_is_R3, bounds_other) - thresh,
                    fixed_val_opt + 0.01, 99.9)
    except (ValueError, RuntimeError):
        hi = 100.0

    pl_cis[name] = (lo, hi)
    opt_val = R3_opt if fixed_is_R3 else R5_opt
    print(f"PL 95% CI  {name}: [{lo:.3f}, {hi:.3f}]  (optimum = {opt_val:.3f})")

In [ ]:
# ── CIs for ALL fluxes from PL bounds on [R3, R5] ────────────────────────────
# Since each flux is LINEAR in (R3, R5), the PL CI for flux f = a0 + a1*R3 + a2*R5
# is obtained by optimising f over the joint PL contour.  For marginal 1-D PL CIs
# this equals   f_opt ± sqrt(chi2_95_1df) * sqrt(a @ cov_r3r5 @ a)
z_pl = np.sqrt(scipy.stats.chi2.ppf(0.95, 1))  # ~1.96

ci_lo_all = net_opt - z_pl * std_all_net
ci_hi_all = net_opt + z_pl * std_all_net

df_all = pd.DataFrame({
    "flux"   : [n[:-2] for n in net_names],   # strip ".n"
    "optimum": net_opt,
    "CI lo"  : ci_lo_all,
    "CI hi"  : ci_hi_all,
}).set_index("flux")

print("95% profile-likelihood CIs for all net fluxes:")
print(df_all.round(3).to_string())

In [ ]:
# ── Summary plot ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

flux_labels = df_all.index.tolist()
y   = np.arange(len(flux_labels))
opt = df_all["optimum"].values
lo  = df_all["CI lo"].values
hi  = df_all["CI hi"].values
err = np.vstack([opt - lo, hi - opt])

ax.barh(y, opt, xerr=err, color="steelblue", alpha=0.7,
        error_kw=dict(ecolor="black", capsize=4, linewidth=1.5))
ax.set_yticks(y)
ax.set_yticklabels(flux_labels)
ax.axvline(0, color="k", linewidth=0.8, linestyle="--")
ax.set_xlabel("Net flux (a.u.)")
ax.set_title("All net fluxes — optimum ± 95% PL CI")
plt.tight_layout()
plt.savefig("all_flux_cis.png", dpi=150)
plt.show()
print("Saved: all_flux_cis.png")